# Basket Analysis for Product Co-Occurrence

In [ ]:
# Order log data to be analyzed
# Includes invalid data,
# such as entries with an product_id of None
# or a quantity less than 1
purchase_logs = [
    {"order_id": "1001", "product_id": "Prod_A", "quantity": 1},
    {"order_id": "1001", "product_id": "Prod_B", "quantity": 1},
    {"order_id": "1002", "product_id": "Prod_A", "quantity": 2},
    {"order_id": "1002", "product_id": "Prod_C", "quantity": 1},
    {"order_id": "1003", "product_id": "Prod_B", "quantity": 1},
    {"order_id": "1003", "product_id": "Prod_C", "quantity": 1},
    {"order_id": "1004", "product_id": "Prod_A", "quantity": 1},
    {"order_id": "1004", "product_id": "Prod_B", "quantity": 1},
    {"order_id": "1004", "product_id": "Prod_C", "quantity": 1},
    {"order_id": "1005", "product_id": None,     "quantity": 1},  # falsified data
    {"order_id": "1005", "product_id": "Prod_A", "quantity": 0},  # falsified data
    {"order_id": "1005", "product_id": "Prod_D", "quantity": 1},
    {"order_id": "1006", "product_id": "Prod_A", "quantity": 1},
    {"order_id": "1006", "product_id": "Prod_B", "quantity": 2},
    {"order_id": "1007", "product_id": "Prod_A", "quantity": 1},
    {"order_id": "1007", "product_id": "Prod_D", "quantity": 1},
    {"order_id": "1008", "product_id": "Prod_B", "quantity": 1},
    {"order_id": "1008", "product_id": "Prod_C", "quantity": 1},
]

# Data cleansing
def is_valid(log):
    """Verify whether the order log is active
    Args:
        log: Order log dictionary. Contains order_id, product_id, and quantity.
    Returns:
        bool: Returns True if the product_id is not None and the quantity is 1 or greater
    """
    return log["product_id"] is not None and log["quantity"] >= 1

purchase_logs = [log for log in purchase_logs if is_valid(log)]

orders = {}
for log in purchase_logs:
    order_id = log["order_id"]
    product_id = log["product_id"]

    if order_id not in orders:
        orders[order_id] = []

    orders[order_id].append(product_id)

def  analyze_co_occurrence(logs, target_product):
    """
    Calculate the Confidence for product pairs.
    Measures the probability of each product being purchased,
    given that target_product has already been purchased.
    Args:
        logs: dict mapping order_id to list of product_ids
        target_product: product_id used as the starting point for analysis
    Returns:
        A list of tuples (product_id, cross-selling probability)
        sorted in descending order of cross-selling probability
    """
    target_order_count = 0    # number of orders containing the target_product
    co_occurrence_count = {}    # number of items purchased together
    for order, products in orders.items():
        if target_product in products:
            target_order_count += 1
            for product in products:
                if product != target_product:
                    co_occurrence_count[product] = co_occurrence_count.get(product, 0) + 1

    co_occurrence_prob = {}
    for product, count in co_occurrence_count.items():
        co_occurrence_prob[product] = (co_occurrence_count[product] / target_order_count
                                       if  target_order_count > 0 else 0)
        
    sorted_result = sorted(co_occurrence_prob.items(), key=lambda x: x[1], reverse=True)
    return sorted_result


if __name__ == "__main__":
    target = "Prod_A"
    result = analyze_co_occurrence(orders, target)
    for product_id, prob in result:
        print(f"Product:{product_id} | Confidence: {prob:.2%}")        

Product:Prod_B | Confidence: 60.00%
Product:Prod_C | Confidence: 40.00%
Product:Prod_D | Confidence: 20.00%
